# Testes de Requisições — Serviço RAG HighBond (REST + MCP HTTP)

Antes de executar as células abaixo, inicie o servidor em um terminal:
```powershell
.\.venv\Scripts\rag-api.exe
```

In [ ]:
import json
import requests

# URL em Produção (Google Cloud Run) ou Local (http://127.0.0.1:8000)
BASE_URL = "https://rag-highbond-764415736.us-central1.run.app"

# Chave configurada no Cloud Run (RAG_API_KEY):
API_KEY = "hb-rag-secreto-2026"

HEADERS = {"Authorization": f"Bearer {API_KEY}"} if API_KEY else {}

## 1. Health Check (`GET /health`)
Verifica se o Ollama está rodando, se o índice foi carregado, os limites de taxa e as estatísticas do cache LRU.

In [ ]:
r = requests.get(f"{BASE_URL}/health", timeout=10)
print(f"Status: {r.status_code} | Tempo: {r.headers.get('X-Process-Time-Ms')} ms")
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

## 2. Busca Semântica em Português (`POST /buscar`)
Envie uma dúvida em português e veja os endpoints mais relevantes da documentação em inglês.

In [ ]:
payload = {
    "pergunta": "Como atualizar um teste de controle (control test) ou plano de teste?",
    "quantidade": 5,
    "similaridade_minima": 0.55
}

r_busca = requests.post(f"{BASE_URL}/buscar", json=payload, headers=HEADERS, timeout=20)
print(f"Status: {r_busca.status_code} | Tempo no servidor: {r_busca.headers.get('X-Process-Time-Ms')} ms\n")

resultados = r_busca.json()
for item in resultados:
    print(f"#{item['posicao']} [sim={item['similaridade']}] {item['id']}")
    print(f"   Título: {item['title']}")
    print(f"   Trecho: {item['trecho'][:140]}...\n")

## 3. Testando o Cache de Latência (Repetindo a mesma pergunta)
Como o servidor possui cache LRU de embeddings, repetir uma pergunta já feita deve responder em **~1 ms** sem acionar o Ollama novamente.

In [ ]:
r_cache = requests.post(f"{BASE_URL}/buscar", json=payload, headers=HEADERS, timeout=20)
print(f"Tempo com Cache Hit: {r_cache.headers.get('X-Process-Time-Ms')} ms")
print("Estatísticas do cache agora:", requests.get(f"{BASE_URL}/health").json()["cache_embeddings"])

## 4. Obtendo a Documentação Completa do Endpoint (`GET /chunks/{chunk_id}`)
Pega o `id` do 1º resultado da busca acima e traz o Markdown completo com parâmetros, corpo da requisição e exemplos de resposta.

In [ ]:
if resultados:
    melhor_id = resultados[0]["id"]
    r_chunk = requests.get(f"{BASE_URL}/chunks/{melhor_id}", headers=HEADERS, timeout=10)
    chunk = r_chunk.json()
    print(f"ID: {chunk['id']}")
    print(f"Título: {chunk['title']}")
    print(f"Tags: {chunk['tags']}")
    print("-" * 60)
    print(chunk["conteudo"][:1500])

## 5. Bateria com Várias Perguntas (Casos de Uso Reais)

In [ ]:
perguntas = [
    "Como criar um risco estratégico (strategy risk) na organização?",
    "Quais endpoints servem para gerenciar storyboards?",
    "Como deletar uma métrica associada a um risco estratégico?",
    "Como atualizar o papel (role) de um usuário em strategy?",
]

for p in perguntas:
    resp = requests.post(
        f"{BASE_URL}/buscar",
        json={"pergunta": p, "quantidade": 2, "similaridade_minima": 0.5},
        headers=HEADERS,
        timeout=20,
    )
    top = resp.json()
    print(f"Pergunta: {p} ({resp.headers.get('X-Process-Time-Ms')} ms)")
    for r_item in top:
        print(f"  -> [{r_item['similaridade']}] {r_item['title']} ({r_item['id']})")
    print()

## 6. Testando o Endpoint MCP Remoto (`POST /mcp` via Streamable HTTP)
Simula exatamente a chamada que o ChatGPT Connectors, Claude ou VS Code fazem para listar e executar as tools do seu MCP.

In [ ]:
mcp_headers = {
    **HEADERS,
    "Accept": "application/json, text/event-stream",
    "Content-Type": "application/json",
}

# 6.1 Listar as tools disponíveis no servidor MCP
req_list_tools = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/list",
    "params": {}
}
r_mcp_list = requests.post(f"{BASE_URL}/mcp", json=req_list_tools, headers=mcp_headers, timeout=10)
print("Status MCP tools/list:", r_mcp_list.status_code)
print(r_mcp_list.text[:800])

# 6.2 Chamar a tool buscar_highbond via protocolo MCP
req_call_tool = {
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/call",
    "params": {
        "name": "buscar_highbond",
        "arguments": {
            "pergunta": "criar storyboard",
            "quantidade": 2
        }
    }
}
r_mcp_call = requests.post(f"{BASE_URL}/mcp", json=req_call_tool, headers=mcp_headers, timeout=20)
print("\nStatus MCP tools/call:", r_mcp_call.status_code)
print(r_mcp_call.text[:1000])